## Heart Disease Prediction
Dataset: UCI Cleveland Heart Disease

---

In [24]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

os.makedirs('../models', exist_ok=True)
os.makedirs('../data', exist_ok=True)
os.makedirs('../results', exist_ok=True)

In [25]:
# 1.DATA LOADING

def load_heart_data():
    urls = [
        "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data",
        "https://raw.githubusercontent.com/datasets/heart-disease/main/processed.cleveland.data",
    ]
    columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
               'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
    
    for url in urls:
        try:
            df = pd.read_csv(url, header=None, names=columns, na_values='?')
            df.to_csv('../data/heart.csv', index=False)
            print(f"[Data] Downloaded from: {url}")
            return df
        except Exception as e:
            print(f"[Data] Failed: {url} - {e}")
            continue

df = load_heart_data()
print(f"\n[1] Data Loaded: {df.shape[0]} samples, {df.shape[1]-1} features")

[Data] Downloaded from: https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data

[1] Data Loaded: 303 samples, 13 features


In [26]:
# 2. DATA CLEANING & TARGET BINARIZATION
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)
print(f"[2] Target Distribution: {dict(df['target'].value_counts().sort_index())}")

# 3. MISSING VALUE HANDLING
print(f"\n[3] Missing Values Before deletion: {df.isnull().sum().sum()}")
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
print(f"    Missing Values After deletion: {df.isnull().sum().sum()}")

# 4. FEATURES
key_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
print(f"\n[4] Key Features: {key_features}")

[2] Target Distribution: {0: np.int64(164), 1: np.int64(139)}

[3] Missing Values Before deletion: 6
    Missing Values After deletion: 0

[4] Key Features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']


In [27]:
# ============================================================
# 5. SCALING & SPLIT
# ============================================================

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n[5] Train/Test Split: {len(X_train)} / {len(X_test)}")


joblib.dump(scaler, '../models/heart_scaler.pkl')
print("    Scaler saved: ../models/heart_scaler.pkl")


[5] Train/Test Split: 242 / 61
    Scaler saved: ../models/heart_scaler.pkl


In [28]:
# ============================================================
# 6. MODEL TRAINING
# ============================================================

results = []

def evaluate_model(name, model, X_test, y_test, is_torch=False):
    if is_torch:
        model.eval()
        with torch.no_grad():
            outputs = model(torch.FloatTensor(X_test))
            y_pred = (outputs.squeeze() > 0.5).float().numpy()
            y_prob = outputs.squeeze().numpy()
    else:
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_prob)

    print(f"\n{name} Results:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc:.4f}")
    print(f"  Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

    return {
        'Disease': 'Heart',
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'ROC-AUC': round(roc, 4)
    }

print("\n" + "=" * 40)
print("TRAINING MODELS")
print("=" * 40)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
results.append(evaluate_model('Logistic Regression', lr, X_test_scaled, y_test))

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
results.append(evaluate_model('Random Forest', rf, X_test_scaled, y_test))

# PyTorch MLP
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_train_scaled.shape[1]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train.values).unsqueeze(1))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print("\nTraining PyTorch MLP...")
model.train()
for epoch in range(200):
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1}/200, Loss: {epoch_loss/len(train_loader):.4f}")

results.append(evaluate_model('MLP', model, X_test_scaled, y_test, is_torch=True))


TRAINING MODELS

Logistic Regression Results:
  Accuracy:  0.8689
  Precision: 0.8125
  Recall:    0.9286
  F1-Score:  0.8667
  ROC-AUC:   0.9513
  Confusion Matrix:
[[27  6]
 [ 2 26]]

Random Forest Results:
  Accuracy:  0.8852
  Precision: 0.8182
  Recall:    0.9643
  F1-Score:  0.8852
  ROC-AUC:   0.9513
  Confusion Matrix:
[[27  6]
 [ 1 27]]

Training PyTorch MLP...
  Epoch 50/200, Loss: 0.2444
  Epoch 100/200, Loss: 0.1557
  Epoch 150/200, Loss: 0.1627
  Epoch 200/200, Loss: 0.0811

MLP Results:
  Accuracy:  0.8197
  Precision: 0.7297
  Recall:    0.9643
  F1-Score:  0.8308
  ROC-AUC:   0.9394
  Confusion Matrix:
[[23 10]
 [ 1 27]]


In [29]:
# ============================================================
# 7. EXPORT BEST MODEL
# ============================================================
print("\n" + "=" * 40)
print("MODEL COMPARISON & EXPORT")
print("=" * 40)

results_df = pd.DataFrame(results)
print("\n", results_df.to_string(index=False))

best_idx = results_df['ROC-AUC'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"\nBest Model: {best_model_name} (ROC-AUC: {results_df.loc[best_idx, 'ROC-AUC']})")

if best_model_name == 'Logistic Regression':
    joblib.dump(lr, '../models/heart_model.pkl')
elif best_model_name == 'Random Forest':
    joblib.dump(rf, '../models/heart_model.pkl')
else:
    torch.save({
        'input_dim': X_train_scaled.shape[1],
        'state_dict': model.state_dict()
    }, '../models/heart_model.pth')

print(f"    Best model saved to ../models/heart_model.*")
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/heart_results.csv', index=False)
print("    Results saved to ../results/heart_results.csv")


MODEL COMPARISON & EXPORT

 Disease               Model  Accuracy  Precision  Recall     F1  ROC-AUC
  Heart Logistic Regression    0.8689     0.8125  0.9286 0.8667   0.9513
  Heart       Random Forest    0.8852     0.8182  0.9643 0.8852   0.9513
  Heart                 MLP    0.8197     0.7297  0.9643 0.8308   0.9394

Best Model: Logistic Regression (ROC-AUC: 0.9513)
    Best model saved to ../models/heart_model.*
    Results saved to ../results/heart_results.csv
